# GPT 4.1 mini

# Severity Verification, Detection Controls verification , DFMEA PFMEA gap analysis



# severity updated

In [ ]:
import os
import re
import copy
import time
import webbrowser
import openpyxl
import pandas as pd
from openpyxl.styles import Alignment, Font
from dotenv import load_dotenv
from openai import OpenAI
from xlsx2html import xlsx2html
from openpyxl.utils import get_column_letter

load_dotenv() 

client = OpenAI(
    api_key=os.getenv("API_KEY_S30"),
    #api_key=os.getenv("API_KEY_Pro"),
    base_url="https://api.euron.one/api/v1/euri",
)

def extract_number(val):
    if val is None: return None
    if isinstance(val, (int, float)): return int(val)
    matches = re.findall(r'\d+', str(val))
    return int(matches[0]) if matches else None

def get_det_zone(sev, det):
    if sev is None or det is None: return ""
    try: s, d = int(sev), int(det)
    except ValueError: return ""
    if not (1 <= s <= 10 and 1 <= d <= 10): return ""
    matrix = {
        10: {1:3, 2:2, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        9:  {1:3, 2:2, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        8:  {1:3, 2:2, 3:2, 4:2, 5:2, 6:2, 7:1, 8:1, 9:1, 10:1},
        7:  {1:3, 2:3, 3:3, 4:2, 5:2, 6:2, 7:2, 8:1, 9:1, 10:1},
        6:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:2, 7:2, 8:1, 9:1, 10:1},
        5:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:2, 9:2, 10:2},
        4:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:2, 9:2, 10:2},
        3:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:3, 9:3, 10:3},
        2:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:3, 9:3, 10:3},
        1:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:3, 9:3, 10:3}
    }
    return matrix[d][s]

def get_sev_zone(sev, occ):
    if sev is None or occ is None: return ""
    try: s, o = int(sev), int(occ)
    except ValueError: return ""
    if not (1 <= s <= 10 and 1 <= o <= 10): return ""
    matrix = {
        10: {1:3, 2:1, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        9:  {1:3, 2:1, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        8:  {1:3, 2:2, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        7:  {1:3, 2:2, 3:2, 4:2, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        6:  {1:3, 2:2, 3:2, 4:2, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        5:  {1:3, 2:3, 3:2, 4:2, 5:2, 6:2, 7:1, 8:1, 9:1, 10:1},
        4:  {1:3, 2:3, 3:3, 4:3, 5:2, 6:2, 7:1, 8:1, 9:1, 10:1},
        3:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:2, 8:2, 9:1, 10:1},
        2:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:2, 8:2, 9:1, 10:1},
        1:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:3, 9:3, 10:3}
    }
    return matrix[o][s]

def get_priority_level(sev_zone, det_zone):
    if sev_zone is None or det_zone is None: return ""
    try: sz, dz = int(sev_zone), int(det_zone)
    except ValueError: return ""
    if not (1 <= sz <= 3 and 1 <= dz <= 3): return ""
    matrix = {
        3: {1: 2, 2: 2, 3: 3},
        2: {1: 1, 2: 2, 3: 3},
        1: {1: 1, 2: 1, 3: 2}
    }
    return matrix[dz][sz]

def evaluate_severity_with_llm(effect_text, original_sev):
    if not effect_text or str(effect_text).strip().lower() == 'nan': return "", "", 0, 0
    try: clean_original_sev = int(float(original_sev))
    except (ValueError, TypeError): clean_original_sev = str(original_sev).strip()

    system_prompt = """
    You are an expert AIAG PFMEA auditor. Your strict task is to determine the TRUE Severity score based ONLY on the English text describing the failure effect.

    AIAG SEVERITY DEFINITIONS:
    10: Product: Affects safe operation and/or involves noncompliance with regulations without warning. Process: May endanger operator, machine or assembly without warning.
    9: Product: Affects safe operation and/or involves noncompliance with regulations with warning. Process: May endanger operator, machine or assembly with warning.
    8: Product: Loss of primary function (product inoperable, does not affect safe operation). Process: 100% of product may have to be scrapped. Line shutdown or stop ship.
    7: Product: Degradation of primary function (product operable, but at a reduced level of performance). Process: A portion of the production run may have to be scrapped. Deviation from primary process; decreased line speed or added manpower.
    6: Product: Loss of secondary function (product operable but service life greatly reduced, convenience item(s) inoperable, customer dissatisfied). Process: 100% of production run may have to be reworked off line and accepted.
    5: Product: Degradation of secondary function (product operable but appearance affected, convenience item(s) operable at a reduced level, customer dissatisfied. Process: A proportion of the production run may have to be reworked off line and accepted.
    4: Product: Appearance, fit and finish type items do not conform, defect noticed by most of the customers (>75%). Process: 100% of production run may have to be reworked in station before it is processed.
    3: Product: Appearance, fit and finish type items do not conform, defect noticed by about half of the customers (50%). Process: A proportion of the production run may have to be reworked in station before it is processed.
    2: Product: Appearance, fit and finish type items do not conform, defect noticed by discriminating customers (<25%). Process: Slight inconvenience to process, operation or operator.
    1: Product: No discernible effect. Process: No discernible effect.

    RULES:
    1. Ignore any numbers written in parentheses (e.g., "(4)") inside the text when determining the TRUE severity.
    2. If multiple effects are described, the TRUE severity is the HIGHEST AIAG number described.
    3. Compare your TRUE severity to the provided "Stated Severity" variable.
    4. Output your response EXACTLY in this format on two lines:
    Severity: <integer 1-10>
    Reason: <If your TRUE severity matches the Stated Severity, write "severity is correct". If it does not match, explain exactly why you chose a different severity based on the AIAG definitions.>

    LEARNING EXAMPLES:
    Text: "S: Portion of production run may have to be scrapped (5) \n OEM: Portion of production run may have to be scrapped (7)"
    Stated Severity: 7
    Output:
    Severity: 7
    Reason: severity is correct

    Text: "MismatchEU: Loss of primary function (product inoperable, does not affect safe operation) (4)"
    Stated Severity: 4
    Output:
    Severity: 8
    Reason: 'Loss of primary function' maps to AIAG 8. The stated (4) is an under-rating.

    Text: "S: A proportion of the production run may have to be reworked off line and accepted (5)\nOEM: No discernible effect (1)"
    Stated Severity: 5
    Output:
    Severity: 5
    Reason: severity is correct
    (Note: 'A proportion' reworked off line is Severity 5. Do not confuse with '100%' which is Severity 6).

    IMPORTANT FOR VAGUE TEXTS: 
    Never default to Severity 1 for vague failure texts like "Reduced performance" or "Poor life". Vague texts indicating reduced performance or degradation must map closest to Severity 7 (Degradation of primary function). Do not predict 1 unless the text explicitly states "No discernible effect".
    """
    try:
        response = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Stated Severity: {clean_original_sev}\nText: {effect_text}\nOutput:"}],
            temperature=0.0
        )
        output_text = response.choices[0].message.content.strip()
        severity, reason = "", ""
        for line in output_text.split('\n'):
            if line.startswith("Severity:"): severity = line.replace("Severity:", "").strip()
            elif line.startswith("Reason:"): reason = line.replace("Reason:", "").strip()
        return severity, reason, response.usage.prompt_tokens, response.usage.completion_tokens
    except Exception as e:
        return "", f"Error: {str(e)}", 0, 0

def build_det_prompt(file_path):
    xls = pd.ExcelFile(file_path)
    if "P-DET" in xls.sheet_names: df = pd.read_excel(xls, "P-DET")
    elif "p-det" in xls.sheet_names: df = pd.read_excel(xls, "p-det")
    else: return "AIAG DETECTION DEFINITIONS NOT FOUND IN EXCEL."
    prompt = "AIAG DETECTION DEFINITIONS:\n"
    for i in range(len(df)):
        rank, cat, crit = str(df.iloc[i, 1]).strip(), str(df.iloc[i, 2]).strip(), str(df.iloc[i, 3]).strip()
        if rank.isdigit(): prompt += f"- {rank} ({cat}): {crit}\n"
    return prompt

def extract_strongest_detection(det_text):
    if not det_text: return None
    lowest_val = 999
    matches = re.findall(r"\((\d+)\)", str(det_text)) or re.findall(r"\b(\d+)\b", str(det_text))
    if matches:
        for m in matches:
            if int(m) < lowest_val: lowest_val = int(m)
    return lowest_val if lowest_val != 999 else None

def evaluate_detection_with_llm(full_det_text, original_det, pred_det, det_definitions):
    if not full_det_text: return "", 0, 0
    try: clean_original_det = int(float(original_det))
    except (ValueError, TypeError): clean_original_det = str(original_det).strip()
    
    system_prompt = f"""
    You are an expert AIAG PFMEA auditor. Your job is to strictly verify the consistency of the 'Current Process Detection Controls' column against the assigned Detection rankings.

    {det_definitions}

    INSTRUCTIONS FOR AUDIT:
    I will provide you with:
    1. Original Stated Detection: (Variable A)
    2. Strongest Detection Extracted: (Variable B)
    3. The Full Detection Controls Text

    You must perform two checks:
    CHECK 1: Does Variable A exactly match Variable B?
    CHECK 2: SEMANTIC VERIFICATION (CRITICAL). Read the actual words in "Full Detection Controls Text". Do they describe the AIAG criteria for the lowest number written next to them? 
    - If the text describes a weak control (e.g., "visual inspection", "random audit") but claims a strong number (e.g., 2, 3, 4), Check 2 FAILS (Over-rating).
    - If the text describes a strong control (e.g., "automated lock", "error proofing") but claims a weak number (e.g., 7, 8), Check 2 FAILS (Under-rating).

    CRITICAL OUTPUT CONSTRAINTS:
    You must format your response EXACTLY like one of the following two templates.
    TEMPLATE 1 (Pass): "detection controls text matches AIAG detection criteria"
    TEMPLATE 2 (Fail): "detection controls text keywords do not match AIAG detection criteria. [State exactly what failed]. Recommended Detection: [Insert the CORRECT AIAG number]"

    LEARNING EXAMPLES:
    Original Stated Detection: 8
    Strongest Detection Extracted: 8
    Full Detection Controls Text: "Visual inspection by operator (8)"
    Output:
    detection controls text matches AIAG detection criteria

    Original Stated Detection: 3
    Strongest Detection Extracted: 3
    Full Detection Controls Text: "100% visual inspection (3)"
    Output:
    detection controls text keywords do not match AIAG detection criteria. '100% visual inspection' is a subjective visual check, which maps to a higher AIAG number, not 3. Recommended Detection: 8

    Original Stated Detection: 5
    Strongest Detection Extracted: 2
    Full Detection Controls Text: "CMM measurement (5) \\n Error proofing fixture prevents part from being loaded incorrectly (2)"
    Output:
    detection controls text keywords do not match AIAG detection criteria. The strongest control stated is 2 (Error proofing), but the original stated detection is 5. Recommended Detection: 2

    Original Stated Detection: 4
    Strongest Detection Extracted: 4
    Full Detection Controls Text: "Automated camera inspection (4)"
    Output:
    detection controls text matches AIAG detection criteria
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-5.4-mini", 
            messages=[
                {"role": "system", "content": system_prompt}, 
                {"role": "user", "content": f"Original Stated Detection: {clean_original_det}\nStrongest Detection Extracted: {pred_det}\nFull Detection Controls Text:\n{full_det_text}"}
            ],
            temperature=0.0
        )
        return response.choices[0].message.content.strip(), response.usage.prompt_tokens, response.usage.completion_tokens
    except Exception as e:
        return f"Error contacting LLM: {str(e)}", 0, 0

def inject_modern_css(html_path):
    with open(html_path, 'r', encoding='utf-8') as f: html_content = f.read()
    parts = html_content.split('<tr')
    target_idx = next((i for i, part in enumerate(parts) if "PROCESS FAILURE MODE AND EFFECTS ANALYSIS" in part.upper()), -1)
    if target_idx > 1:
        for i in range(1, target_idx): parts[i] = ' class="hidden-row" ' + parts[i]
    html_content = '<tr'.join(parts)
    html_content = re.sub(r'(<td[^>]*>)(.*?PROCESS FAILURE MODE AND EFFECTS ANALYSIS.*?)(</td>)', r'\1<div class="pfmea-main-header">\2</div>\3', html_content, flags=re.IGNORECASE | re.DOTALL)
    modern_css = """<style>@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap'); body { font-family: 'Inter', sans-serif; background-color: #f4f7f9; color: #334155; margin: 40px; } table { border-collapse: collapse; width: 100%; background-color: #ffffff; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); font-size: 14px; } table, th, td { border: 1px solid #e2e8f0 !important; } th, td { padding: 12px 16px !important; line-height: 1.5; } .hidden-row { display: none !important; } .pfmea-main-header { font-size: 24px !important; font-weight: 700 !important; background-color: #1e3a8a !important; color: #ffffff !important; padding: 16px !important; text-align: center !important; border-radius: 6px; text-transform: uppercase; } tr:nth-child(-n+11) td { background-color: #ffffff !important; border: none !important; font-size: 13px; color: #64748b !important; } tr:nth-child(12) td, tr:nth-child(13) td { background-color: #0f172a !important; color: #ffffff !important; font-weight: 600 !important; text-transform: uppercase; font-size: 12px; } tr:nth-child(n+14):nth-child(even) td { background-color: #f8fafc !important; } tr:nth-child(n+14):hover td { background-color: #f1f5f9 !important; transition: background-color 0.2s ease; } td[style*="color: #CC0000"], td[style*="color: CC0000"] { color: #b91c1c !important; background-color: #fef2f2 !important; border: 1px solid #fca5a5 !important; font-size: 14px !important; font-weight: bold !important; }</style>"""
    html_content = html_content.replace('</head>', f'{modern_css}\n</head>') if '</head>' in html_content else modern_css + html_content
    with open(html_path, 'w', encoding='utf-8') as f: f.write(html_content)

def process_and_audit_pfmea(input_file="pfmea_org.xlsx", output_file="pfmea_audited.xlsx"):
    print("Loading workbook and preparing duplicate sheet to preserve formatting...")
    wb = openpyxl.load_workbook(input_file)
    ws = wb["PFMEA"]
    det_definitions_prompt = build_det_prompt(input_file)

    if "PFMEA_Audited" in wb.sheetnames: del wb["PFMEA_Audited"]
    out_ws = wb.copy_worksheet(ws)
    out_ws.title = "PFMEA_Audited"

    for rng in list(out_ws.merged_cells.ranges):
        if rng.min_row >= 12:
            out_ws.unmerge_cells(str(rng))

    total_input_tokens, total_output_tokens = 0, 0

    # 1. Map original columns
    col_map = {}
    for col in range(1, ws.max_column + 1):
        val11 = str(ws.cell(11, col).value or "").strip().lower()
        val12 = str(ws.cell(12, col).value or "").strip().lower()
        val = val11 + " " + val12
        
        if not val.strip(): continue
        
        if "process" in val and "prevention" not in val and "detection" not in val: col_map["process"] = col
        elif "potential failure mode" in val: col_map["potential failure mode"] = col
        elif "effect" in val: col_map["potential effects of failure"] = col
        elif "sev" == val12 or "sev" == val11: col_map["sev"] = col
        elif "cls" == val12 or "cls" == val11: col_map["cls"] = col
        elif "cause" in val or "mechanism" in val: col_map["potential causes / mechanisms of failure"] = col
        elif "prevention" in val: col_map["current process prevention controls"] = col
        elif "occ" == val12 or "occ" == val11: col_map["occ"] = col
        elif "detection controls" in val: col_map["current process detection controls"] = col
        elif "det" == val12 or "det" == val11: col_map["det"] = col 
        elif "rpn" == val12 or "rpn" == val11: col_map["rpn"] = col 
        elif "recommended action" in val or "rec act" in val: col_map["recommended actions"] = col
        elif "severity zone" in val or "sev zone" in val: col_map["severity zone"] = col
        elif "detection zone" in val or "det zone" in val: col_map["detection zone"] = col
        elif "priority level" in val: col_map["priority level"] = col

    # 2. Define Comprehensive Target Columns Order
    targets = [
        ("Process", col_map.get("process")),
        ("Potential Failure Mode", col_map.get("potential failure mode")),
        ("Potential Effects of Failure", col_map.get("potential effects of failure")),
        ("Sev", col_map.get("sev")),
        ("Pred Sev", col_map.get("sev")), 
        ("Sev Reasoning", col_map.get("sev")),
        ("Cls", col_map.get("cls")),
        ("Potential Causes / Mechanisms of Failure", col_map.get("potential causes / mechanisms of failure")),
        ("Current Process Prevention Controls", col_map.get("current process prevention controls")),
        ("Occ", col_map.get("occ")),
        ("Current Process Detection Controls", col_map.get("current process detection controls")),
        ("Det", col_map.get("det")),
        ("Pred Det", col_map.get("det")),
        ("Det Reasoning", col_map.get("det")),
        ("RPN", col_map.get("rpn")),
        ("D-P gap", col_map.get("rpn")),
        ("New RPN", col_map.get("rpn")),
        ("Recommended Actions", col_map.get("recommended actions")),
        ("Severity Zone", col_map.get("severity zone")),
        ("Pred Sev Zone", col_map.get("severity zone") or col_map.get("rpn")),
        ("Detection Zone", col_map.get("detection zone")),
        ("Pred Det Zone", col_map.get("detection zone") or col_map.get("rpn")),
        ("Priority Level", col_map.get("priority level")),
        ("Pred Priority Level", col_map.get("priority level") or col_map.get("rpn")),
    ]

    # 3. Setup clean columns in the duplicated sheet
    for idx, (title, ref_col) in enumerate(targets, 1):
        cell12 = out_ws.cell(12, idx)
        cell13 = out_ws.cell(13, idx)
        cell12.value = title
        out_ws.merge_cells(start_row=12, start_column=idx, end_row=13, end_column=idx)
        
        if ref_col:
            cell12._style = copy.copy(ws.cell(12, ref_col)._style)
            cell13._style = copy.copy(ws.cell(13, ref_col)._style)
            width = ws.column_dimensions[get_column_letter(ref_col)].width
            out_ws.column_dimensions[get_column_letter(idx)].width = width or 15
        else:
            out_ws.column_dimensions[get_column_letter(idx)].width = 15
            
        # Hardcode explicit widths based on column types
        if "Reasoning" in title:
            out_ws.column_dimensions[get_column_letter(idx)].width = 50
        elif title in ["Potential Effects of Failure", "Current Process Prevention Controls"]:
            out_ws.column_dimensions[get_column_letter(idx)].width = 50
        elif title == "D-P gap":
            if col_map.get("process"):
                out_ws.column_dimensions[get_column_letter(idx)].width = ws.column_dimensions[get_column_letter(col_map.get("process"))].width or 15
            else:
                out_ws.column_dimensions[get_column_letter(idx)].width = 15
                
        cell12.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    # 4. Safely drop extra leftover columns beyond our target mapping
    if out_ws.max_column > len(targets):
        out_ws.delete_cols(len(targets) + 1, out_ws.max_column - len(targets))

    # 5. Process Data Rows
    print("Evaluating rows...")
    for row in range(14, ws.max_row + 1):
        # Fetch base data
        o_s = extract_number(ws.cell(row, col_map.get("sev", 0)).value if col_map.get("sev") else None)
        o_o = extract_number(ws.cell(row, col_map.get("occ", 0)).value if col_map.get("occ") else None)
        o_d = extract_number(ws.cell(row, col_map.get("det", 0)).value if col_map.get("det") else None)
        
        effect_text = ws.cell(row, col_map.get("potential effects of failure", 0)).value if col_map.get("potential effects of failure") else None
        det_text = ws.cell(row, col_map.get("current process detection controls", 0)).value if col_map.get("current process detection controls") else None
        
        # Predict LLM Checks
        pred_sev, sev_reasoning = None, ""
        if effect_text and o_s is not None:
            pred_sev_str, sev_reasoning, s_in, s_out = evaluate_severity_with_llm(effect_text, o_s)
            pred_sev = extract_number(pred_sev_str)
            total_input_tokens += s_in
            total_output_tokens += s_out
            
        pred_det, det_reasoning = None, ""
        if det_text and o_d is not None:
            ext_det = extract_strongest_detection(det_text)
            det_reasoning, d_in, d_out = evaluate_detection_with_llm(det_text, o_d, ext_det, det_definitions_prompt)
            match = re.search(r"Recommended Detection:\s*(\d+)", det_reasoning, re.IGNORECASE)
            pred_det = int(match.group(1)) if match else ext_det
            total_input_tokens += d_in
            total_output_tokens += d_out

        p_s = pred_sev if pred_sev is not None else o_s
        p_d = pred_det if pred_det is not None else o_d

        # D-P Gap logic
        dp_gap_val, dp_gap_flag = "", False
        if p_s in [9, 10]:
            if p_d is not None and p_d <= 3:
                dp_gap_val = "no gap"
            else:
                dp_gap_val = "Gap identified. Det must be <3"
                dp_gap_flag = True
        elif p_s is not None and p_s > 0:
            dp_gap_val = "no gap"

        # Write clean data directly matching the columns
        for idx, (title, ref_col) in enumerate(targets, 1):
            cell = out_ws.cell(row, idx)
            
            # Inherit cell style of the original column (background, borders)
            if ref_col:
                cell._style = copy.copy(ws.cell(row, ref_col)._style)
            
            if "Reasoning" in title:
                cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
            else:
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            
            # Fill Logic
            if title == "Pred Sev":
                cell.value = pred_sev if pred_sev is not None else ""
                if sev_reasoning and "severity is correct" not in sev_reasoning.lower(): cell.font = Font(color="CC0000")
            elif title == "Sev Reasoning":
                cell.value = sev_reasoning
                if sev_reasoning and "severity is correct" not in sev_reasoning.lower(): cell.font = Font(color="CC0000")
            elif title == "Pred Det":
                cell.value = pred_det if pred_det is not None else ""
                if "keywords do not match" in det_reasoning.lower(): cell.font = Font(color="CC0000")
            elif title == "Det Reasoning":
                cell.value = det_reasoning
                if "keywords do not match" in det_reasoning.lower(): cell.font = Font(color="CC0000")
            elif title == "D-P gap":
                cell.value = dp_gap_val
                if dp_gap_flag: cell.font = Font(color="CC0000")
            elif title == "RPN":
                cell.value = (o_s * o_o * o_d) if (o_s is not None and o_o is not None and o_d is not None) else None
            elif title == "New RPN":
                cell.value = (p_s * o_o * p_d) if (p_s is not None and o_o is not None and p_d is not None) else None
            elif title == "Pred Sev Zone":
                zone = get_sev_zone(p_s, o_o)
                cell.value = zone
                orig_zone = extract_number(ws.cell(row, col_map.get("severity zone", 0)).value if col_map.get("severity zone") else None)
                if orig_zone is not None and orig_zone != zone: cell.font = Font(color="CC0000")
            elif title == "Pred Det Zone":
                zone = get_det_zone(p_s, p_d)
                cell.value = zone
                orig_zone = extract_number(ws.cell(row, col_map.get("detection zone", 0)).value if col_map.get("detection zone") else None)
                if orig_zone is not None and orig_zone != zone: cell.font = Font(color="CC0000")
            elif title == "Pred Priority Level":
                sz = get_sev_zone(p_s, o_o)
                dz = get_det_zone(p_s, p_d)
                prio = get_priority_level(sz, dz)
                cell.value = prio
                orig_prio = extract_number(ws.cell(row, col_map.get("priority level", 0)).value if col_map.get("priority level") else None)
                if orig_prio is not None and orig_prio != prio: cell.font = Font(color="CC0000")
            else:
                cell.value = ws.cell(row, ref_col).value if ref_col else None

    wb.save(output_file)
    wb.close()
    
    print(f"\nAuditing complete. Saved clean output to {output_file} under sheet 'PFMEA_Audited'.")
    print(f"Total Input Tokens: {total_input_tokens} | Output Tokens: {total_output_tokens}")

    print("Generating HTML report...")
    output_html = "audited_pfmea.html"
    time.sleep(1) 
    xlsx2html(output_file, output_html, sheet="PFMEA_Audited")
    
    print("Applying modern aesthetics to HTML...")
    inject_modern_css(output_html)
    
    html_abs_path = os.path.abspath(output_html)
    webbrowser.open(f"file://{html_abs_path}")
    print("Opened audited PFMEA in default browser.")

if __name__ == "__main__":
    process_and_audit_pfmea()

Loading workbook and preparing duplicate sheet to preserve formatting...
Evaluating rows...

Auditing complete. Saved clean output to pfmea_audited.xlsx under sheet 'PFMEA_Audited'.
Total Input Tokens: 15365 | Output Tokens: 980
Generating HTML report...
Applying modern aesthetics to HTML...
Opened audited PFMEA in default browser.
